In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from numpy.random import RandomState
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import math
from sklearn.linear_model import LogisticRegression
from math import log
from imblearn.metrics import geometric_mean_score
from sklearn.metrics import accuracy_score,f1_score
from sklearn import svm,calibration
from sklearn.svm import SVC
from sklearn.utils import shuffle
from scipy.spatial import distance
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from FairKMeans import FairKMeans

In [ ]:
dataset=pd.read_csv("compass_dataset_clean.csv")
dataset=shuffle(dataset)

In [ ]:
dataset.dropna()

In [ ]:
#scaling
min_max_scaler=preprocessing.MinMaxScaler()
x=dataset.values
x_scaled=min_max_scaler.fit_transform(x)
dfX = pd.DataFrame(x_scaled)
dfX.columns=dataset.columns
X_Final=dfX
index=list()
for i in range(0,len(X_Final)):
    index.append(i)

X_Final['index']=index

In [ ]:
#create training, test and unlabeled data
train_len=len(dataset)*0.1
test_len=len(dataset)*0.2
unlabel_len=len(dataset)*0.7
print(train_len)
print(test_len)
print(unlabel_len)

In [ ]:
#take initial sample
initial_label=X_Final.sample(n=round(train_len),random_state=101)
#delete from unlabel
unlabel=X_Final.drop(initial_label.index)
test_dataset=unlabel.sample(n=round(test_len),replace=False)
unlabel=unlabel.drop(test_dataset.index)

In [ ]:
len(initial_label)

In [ ]:
clf=LogisticRegression(penalty='l2',solver='liblinear')


In [ ]:
k=180

In [ ]:
minority_sample=list()
gmean_score=list()
acc_score=list()
total_attribute=list()
samples=list()
f1_scr=list()

for i in range(0,11):
    fair_cluster=pd.DataFrame()
    print("iteration ",i)
    X_train=initial_label.drop(['label'],axis=1)
    y_train=initial_label['label']
    clf.fit(X_train,y_train)
    X_U=unlabel.drop(['label'],axis=1)
    
    #predict
    print("predict ")
    probs=clf.predict_proba(X_U)
    #calculate the entropy
    entropy=list()
    for p in probs:
        ent=0
        ent1=-p[0] * log(p[0],2)
        ent2=-p[1] * log(p[1],2)
        ent=ent1+ent2
        entropy.append(ent)
    X_cluster=pd.DataFrame(X_U)
    X_cluster['entropy']=entropy
    
    #call fairclustering
    fair_kmeans = FairKMeans(n_clusters=k, sensitive_attr=sensitive_attr)
    fair_kmeans.fit(X)
    labels = fair_kmeans.predict(X)
    
    
    data_cluster=unlabel.copy()
    data_cluster['entropy']=entropy
    data_cluster['cluster']=labels
    X_cluster=X_cluster.to_numpy()
    similarity_to_center=list()
    for i, instance in enumerate(X_cluster):
        cluster_label = labels[i]
        centroid = fair_kmeans.cluster_centers_[cluster_label] # cluster center of the cluster of that instance
        similarity=distance.euclidean(instance,centroid)
        similarity_to_center.append(similarity)
    data_cluster['similarity']=similarity_to_center
    cluster_labels=pd.DataFrame({"index":data_cluster['index'],'cluster':labels,
                                    'uncertainty':data_cluster['entropy'],
                                 'similarity':similarity_to_center})
    maximum=data_cluster['similarity'].max()
    minimum=data_cluster['similarity'].min()
    nm=(data_cluster['similarity']-minimum)/(maximum-minimum)
    data_cluster['normalized_similarity']=nm
    #fair_cluster=0
    s=0
    fair=list()
    fair_cluster=pd.DataFrame()
    for i in range(0,k):
        #calculate the fairness score of each cluster
        fair_cluser=fair_kmeans.calculate_fairness_score(i)
    fair_cluster=pd.concat(fair,ignore_index=False)
    
    
    cluster_dict=dict()
    #creating the dataframe for each cluster
    for item in range(0,k):
        cluster_dict['cluster_{0}'.format(item)]=fair_cluster[fair_cluster['cluster']==item] 
    #Create the dataframe for each cluster
    for i in range(0,k):
            globals()['cluster_{}'.format(i)] = pd.DataFrame(cluster_dict['cluster_{}'.format(i)])
    for i in range(0,k):
        globals()['cluster_{}'.format(i)]=globals()['cluster_{}'.format(i)].sort_values(by="normalized_similarity",ascending=False)
    center_data=pd.DataFrame()   
    #calculate the sample representative and uncertainty score
    alpha=0.6
    for i in range(0,k):
        data1=globals()['cluster_{}'.format(i)]
        final_score=alpha*(data1['similarity'])+(1.0-alpha)*data1['entropy']
        globals()['cluster_{}'.format(i)]['final_score']=final_score
        globals()['cluster_{}'.format(i)]=globals()['cluster_{}'.format(i)].sort_values(by='final_score',ascending=False)
    
    
    for i in range(0,k):
            center_data=center_data.append(globals()['cluster_{}'.format(i)].head(1))
    
    
    center_data.drop(['entropy','cluster','similarity','normalized_similarity','final_score'],axis=1,inplace=True)
    
    #add the sample to labeled dataset
    initial_label=initial_label.append(center_data)
    
    #measure the performance
    X=initial_label.drop(['label'],axis=1)
    y=initial_label['label']
    X_test=test_dataset.drop(['label'],axis=1)
    y_test=test_dataset['label']
    
    print("model performance ",i)
    clf.fit(X,y)
    y_pred=clf.predict(X_test)
    y_fair=clf.predict(X)

    gmean=geometric_mean_score(y_pred,y_test)
    f1=f1_score(y_pred,y_test)
    gmean_score.append(gmean)
    f1_scr.append(f1)
    acc=accuracy_score(y_pred,y_test)
    acc_score.append(acc)

    #remove from unlabel
    #unlabel.drop(['entropy','cluster','similarity'],axis=1,inplace=True)
    unlabel=unlabel[~unlabel['index'].isin(center_data['index'])]
    
        

In [ ]:
print("Accuracy over 10 runs ",acc_score)
print("F1 score over 10 runs",f1_scr)
print("GMeans score over 10 runs",gmean_score)

In [ ]:
#add the prediction sample to calculate the fairness
data=pd.DataFrame(initial_label)
data['y_hat']=y_fair
total_0 = data[data.race == 0]
total_1 = data[data.race == 1]

A = data.loc[(data.y_hat == 1) & (data.race == 0)]
B = data.loc[(data.y_hat == 1) & (data.race == 1)]

SP = abs((len(A) / len(total_0)) - (len(B) / len(total_1)))
print("SP", SP)

# calculate Equal opportunity differences
A = data.loc[(data.y_hat == 1) & (data.race == 0) & (data.label == 1)]
B = data.loc[(data.y_hat == 1) & (data.race == 1) & (data.label == 1)]

total_label_1_0 = data[(data.race == 0) & (data.label == 1)]
total_label_1_1 = data[(data.race == 1) & (data.label == 1)]

E_Opp = abs((len(A) / len(total_label_1_0)) - (len(B) / len(total_label_1_1)))
print("E_OPP", E_Opp)

# calculate E_odds
A = data.loc[(data.y_hat == 1) & (data.race == 0) & (data.label == 0)]
A1 = data.loc[(data.y_hat == 1) & (data.race == 1) & (data.label == 0)]
B = data.loc[(data.y_hat == 1) & (data.race == 0) & (data.label == 1)]
B1 = data.loc[(data.y_hat == 1) & (data.race == 1) & (data.label == 1)]

total_label_0_0 = data[(data.race == 0) & (data.label == 0)]
total_label_0_1 = data[(data.race == 1) & (data.label == 0)]

E_odds = abs(((len(A) / len(total_label_0_0)) - (len(A1) / len(total_label_0_1)))*0.5 + ((len(B) / len(total_label_1_0)) - (len(B1) / len(total_label_1_1))*0.5))
print("E_Odd", E_odds)